In [2]:
import copy

import torch_geometric.transforms as T

from qchem.dataset import PGDS
from qchem.model import GraphNet

from qchem_lmm.model import LMMDataset, Llama, LMM


class IndexY(T.BaseTransform):
    def __init__(self, i):
        self.i = i
        
    def __call__(self, data):
        # shallow-copy the data so that we prevent in-place data modification.
        return self.forward(copy.copy(data))
    
    def forward(self, data):
        y = data.y[:,self.i]
        data.y = y
        return data

transforms = T.Compose([IndexY(7)])

ds_param = {'dataset': 'QM9',
            'pg_param': {'root': './data/pyg/qm9/',
                                 'transform': transforms}}
llm_param = {}

gnn_param = {'in_channels': 11, 
             'hidden': 256, 
             'out_channels': 256,
             'depth': 2,
             'convolution': 'SAGEConv',
             'pooling': None,
             'conv_act': 'ReLU',
             'dropout': .3,
             'activation': None,
             'normal': None,
             'layer_param': {'normalize': True},
             'device': 'cuda:0',
             'ffn_param': None}

device = 'cuda:0'

pgds = PGDS(**ds_param)
gnn = GraphNet(gnn_param)
llm = Llama()

lmm_ds = LMMDataset(pgds, llm.tokenizer, training=True)

lmm = LMM(llm, 
          gnn, 
          in_dim=256,        
          hidden_dim=512, 
          out_dim=256, 
          llm_dim=4096, 
          device=device, 
          output_dir='./data')

lmm.train_model(lmm_ds)

#prompt = LMMDataset(pgds, llm.tokenizer, training=False)
#lmm.infer(prompt[0])



creating pytorch geometric QM9 dataset...
GraphNet SAGEConv loaded...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000
